# CrownScaler-style AI Upscaler — Kaggle Edition v1

Два режима: **Basic** для обработки в один клик и **Advanced** для настройки Tile Size, Tile Padding, FP16 и собственной модели.

Поддерживается Real-ESRGAN x4plus и совместимые с его архитектурой RRDBNet `.pth` checkpoint-файлы. Видео обрабатывается покадрово.

**Важно:** произвольные `.pth`, `.safetensors`, `.onnx`, HAT, DAT, SwinIR, SUPIR и TensorRT не являются взаимозаменяемыми. Для каждой архитектуры нужен отдельный загрузчик. В этой первой версии они не заявлены как поддерживаемые.

## 1. Настрой Kaggle
В Notebook settings включи **Accelerator → GPU** и **Internet → On**. Затем запускай ячейки сверху вниз.

In [ ]:
!nvidia-smi
!pip -q install realesrgan basicsr facexlib gfpgan gradio opencv-python-headless

## 2. Импорты и рабочие папки

In [ ]:
import os, cv2, torch, shutil, zipfile, urllib.request, traceback, subprocess
from pathlib import Path
import gradio as gr
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet

ROOT = Path("/kaggle/working/crownscaler_kaggle")
MODEL_DIR, OUTPUT_DIR = ROOT/"models", ROOT/"outputs"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_URL = "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth"

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))

## 3. Загрузка модели и инициализация
Пользовательский checkpoint должен соответствовать архитектуре Real-ESRGAN x4plus RRDBNet. Другие архитектуры этим загрузчиком не поддерживаются.

In [ ]:
def resolve_model(custom_path=""):
    if custom_path and str(custom_path).strip():
        p = Path(str(custom_path).strip())
        if not p.is_file():
            raise FileNotFoundError(f"Не найден файл модели: {p}")
        return str(p)
    p = MODEL_DIR/"RealESRGAN_x4plus.pth"
    if not p.exists():
        urllib.request.urlretrieve(MODEL_URL, p)
    return str(p)

def make_upsampler(model_path, tile=256, tile_pad=10, half=True):
    if not torch.cuda.is_available():
        raise RuntimeError("GPU не обнаружен. Включи GPU в настройках Kaggle.")
    net = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                  num_block=23, num_grow_ch=32, scale=4)
    return RealESRGANer(
        scale=4, model_path=model_path, model=net,
        tile=int(tile), tile_pad=int(tile_pad), pre_pad=0,
        half=bool(half), device=torch.device("cuda")
    )

## 4. Обработка изображений

In [ ]:
def run_images(files, custom_path, tile, tile_pad, half, scale):
    if not files:
        raise gr.Error("Сначала загрузи изображение.")
    model_path = resolve_model(custom_path)
    upsampler = make_upsampler(model_path, tile, tile_pad, half)
    run_dir = OUTPUT_DIR/"images"
    run_dir.mkdir(parents=True, exist_ok=True)
    results, errors = [], []
    for f in files:
        try:
            img = cv2.imread(str(f), cv2.IMREAD_COLOR)
            if img is None:
                raise ValueError("Не удалось прочитать файл")
            out, _ = upsampler.enhance(img, outscale=float(scale))
            dst = run_dir/(Path(f).stem + f"_upscaled_{scale}x.png")
            cv2.imwrite(str(dst), out)
            results.append(str(dst))
        except Exception as e:
            errors.append(f"{Path(f).name}: {e}")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    if not results:
        raise gr.Error("Все файлы завершились ошибкой: " + " | ".join(errors))
    zpath = ROOT/"upscaled_images.zip"
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for p in results:
            z.write(p, arcname=Path(p).name)
    status = f"Готово: {len(results)} файл(ов)."
    if errors:
        status += " Ошибки: " + " | ".join(errors)
    return results, str(zpath), status

## 5. Обработка видео
Видео обрабатывается кадр за кадром, без интерполяции движения. Для экономии VRAM начальный Tile Size установлен на 128.

In [ ]:
def run_video(video_path, custom_path, tile, tile_pad, half, scale):
    if not video_path:
        raise gr.Error("Сначала загрузи видео.")
    model_path = resolve_model(custom_path)
    upsampler = make_upsampler(model_path, tile, tile_pad, half)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise gr.Error("Не удалось открыть видео.")
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    ow, oh = int(w*float(scale)), int(h*float(scale))
    raw = ROOT/"video_intermediate.mp4"
    writer = cv2.VideoWriter(str(raw), cv2.VideoWriter_fourcc(*"mp4v"), fps, (ow, oh))
    if not writer.isOpened():
        cap.release()
        raise gr.Error("Не удалось создать выходной видеофайл.")
    count = 0
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            enhanced, _ = upsampler.enhance(frame, outscale=float(scale))
            if enhanced.shape[:2] != (oh, ow):
                enhanced = cv2.resize(enhanced, (ow, oh), interpolation=cv2.INTER_LANCZOS4)
            writer.write(enhanced)
            count += 1
            if count % 50 == 0:
                print(f"Кадров обработано: {count}/{total or '?'}")
    finally:
        cap.release()
        writer.release()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    final = ROOT/"upscaled_video.mp4"
    ffmpeg = shutil.which("ffmpeg")
    if ffmpeg:
        subprocess.run([ffmpeg, "-y", "-loglevel", "error", "-i", str(raw),
                        "-c:v", "libx264", "-pix_fmt", "yuv420p", "-an", str(final)],
                       check=False)
    if not final.exists() or final.stat().st_size == 0:
        shutil.copy2(raw, final)
    return str(final), f"Обработано кадров: {count}; исходный FPS: {fps:.2f}"

## 6. Интерфейс Basic / Advanced / Video

In [ ]:
with gr.Blocks(theme=gr.themes.Soft(), title="CrownScaler Kaggle Edition") as app:
    gr.Markdown("# 👑 CrownScaler-style AI Upscaler")
    gr.Markdown("Kaggle Edition · Basic / Advanced · Image & Video")

    with gr.Tab("Basic — Images"):
        basic_files = gr.Files(label="Изображения", file_count="multiple",
                               file_types=["image"], type="filepath")
        basic_scale = gr.Radio([2, 4], value=4, label="Масштаб")
        basic_btn = gr.Button("✨ START UPSCALE", variant="primary")
        basic_gallery = gr.Gallery(label="Результаты", columns=2)
        basic_zip = gr.File(label="ZIP с результатами")
        basic_status = gr.Textbox(label="Статус")
        basic_btn.click(lambda fs, sc: run_images(fs, "", 256, 10, True, sc),
                        [basic_files, basic_scale],
                        [basic_gallery, basic_zip, basic_status])

    with gr.Tab("Advanced — Images"):
        adv_files = gr.Files(label="Изображения", file_count="multiple",
                             file_types=["image"], type="filepath")
        custom_path = gr.Textbox(label="Путь к RRDBNet x4plus .pth (необязательно)",
                                 placeholder="/kaggle/input/dataset/model.pth")
        tile = gr.Slider(0, 512, value=256, step=32, label="Tile Size (0 = без тайлов)")
        tile_pad = gr.Slider(0, 64, value=10, step=2, label="Tile Padding")
        half = gr.Checkbox(value=True, label="FP16 — экономия VRAM")
        adv_scale = gr.Radio([2, 4], value=4, label="Масштаб")
        adv_btn = gr.Button("✨ START ADVANCED UPSCALE", variant="primary")
        adv_gallery = gr.Gallery(label="Результаты", columns=2)
        adv_zip = gr.File(label="ZIP с результатами")
        adv_status = gr.Textbox(label="Статус")
        adv_btn.click(run_images,
                      [adv_files, custom_path, tile, tile_pad, half, adv_scale],
                      [adv_gallery, adv_zip, adv_status])

    with gr.Tab("Video"):
        video_in = gr.Video(label="Исходное видео")
        video_model = gr.Textbox(label="Путь к RRDBNet x4plus .pth (необязательно)")
        v_tile = gr.Slider(0, 512, value=128, step=32, label="Tile Size")
        v_pad = gr.Slider(0, 64, value=10, step=2, label="Tile Padding")
        v_half = gr.Checkbox(value=True, label="FP16")
        v_scale = gr.Radio([2, 4], value=4, label="Масштаб")
        video_btn = gr.Button("🎬 UPSCALE VIDEO", variant="primary")
        video_out = gr.Video(label="Результат")
        video_status = gr.Textbox(label="Статус")
        video_btn.click(run_video,
                        [video_in, video_model, v_tile, v_pad, v_half, v_scale],
                        [video_out, video_status])

    gr.Markdown("**Подсказка при CUDA OOM:** уменьши Tile Size: 256 → 128 → 64. Для мелкого текста и игрового интерфейса обязательно проверяй результат: нейросеть может менять детали.")
app.queue().launch(share=True, debug=True)